# 🔗 Défi quotidien — Pipelines LangChain avec LLM open-source (CPU)

Notebook minimal qui s'exécute **de bout en bout sur CPU** avec de petits modèles ouverts.

**Contenu :**
1. Configuration de l'environnement
2. Première chaîne LLM (réécriture de texte)
3. Pipeline en 2 étapes : résumé → 3 points clés
4. **Bonus** : chaîne de conversation avec mémoire

> ⚠️ **Note API importante — à lire.** Le tutoriel d'origine utilise `LLMChain`, `SimpleSequentialChain` et `ConversationChain`. Ces classes ont été **retirées de LangChain 1.x** (2025+). Ce notebook utilise l'**API moderne recommandée** — le pipe `prompt | llm` (Runnables / LCEL) — qui fait exactement la même chose et fonctionne sur les versions actuelles. Une note en bas montre l'équivalence avec l'ancienne syntaxe.

---
## Partie 1 : Configuration de l'environnement

Un runtime **CPU** suffit pour les petits modèles.

In [ ]:
!pip install -q langchain langchain-core langchain-community transformers sentencepiece accelerate

In [ ]:
# (Facultatif) Vérifier le matériel
!nvidia-smi 2>/dev/null || echo "CPU runtime — parfait pour les petits modèles"

import langchain
print("LangChain version :", langchain.__version__)

---
## Partie 2 : Charger un petit modèle ouvert + première chaîne LLM

On utilise **`google/flan-t5-small`** (~80M params, seq2seq, idéal CPU et bon pour suivre des instructions).

Étapes : charger modèle + tokenizer → envelopper dans un `pipeline` HF → envelopper dans `HuggingFacePipeline` → composer avec un `PromptTemplate`.

In [ ]:
import warnings, time
warnings.filterwarnings("ignore")

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
from langchain_community.llms import HuggingFacePipeline
from langchain_core.prompts import PromptTemplate

MODEL_NAME = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

# pipeline HF -> le type 'text2text-generation' correspond aux modèles seq2seq (T5)
hf_pipe = pipeline(
    "text2text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=80,
)

# Wrapper LangChain réutilisé dans tout le notebook
llm = HuggingFacePipeline(pipeline=hf_pipe)
print("✅ Modèle chargé :", MODEL_NAME)

In [ ]:
# Première chaîne : PromptTemplate | llm  (RunnableSequence via l'opérateur pipe)
rewrite_prompt = PromptTemplate.from_template(
    "Rewrite this text to be simpler for beginners: {text}"
)

rewrite_chain = rewrite_prompt | llm

texte = ("The mitochondrion is a double-membraned organelle that generates "
         "most of the cell's supply of adenosine triphosphate, used as chemical energy.")

t0 = time.time()
result = rewrite_chain.invoke({"text": texte})
print("Sortie :", result)
print(f"\n⏱️ Latence : {time.time()-t0:.2f} s")

> 💡 `flan-t5-small` est petit : les réécritures sont parfois courtes ou littérales. Augmentez `max_new_tokens`, ou passez à `google/flan-t5-base` si le runtime le permet, pour une meilleure qualité.

---
## Partie 3 : Pipeline en 2 étapes — Résumé → 3 points clés

On enchaîne deux prompts. La sortie de l'étape 1 (résumé) alimente l'étape 2 (mise en puces).
On utilise `RunnableSequence` construite avec le pipe `|` et un petit adaptateur pour reformater la sortie intermédiaire.

In [ ]:
from langchain_core.runnables import RunnableLambda

# Étape 1 : résumé
summarize_prompt = PromptTemplate.from_template(
    "Summarize the following paragraph in one sentence: {text}"
)

# Étape 2 : 3 points clés à partir du résumé
bullets_prompt = PromptTemplate.from_template(
    "List exactly 3 key bullet points about this: {summary}"
)

# Adaptateur : transforme la string de sortie de l'étape 1 en dict attendu par l'étape 2
to_bullets_input = RunnableLambda(lambda summary: {"summary": summary})

# Pipeline complet : prompt1 -> llm -> adaptateur -> prompt2 -> llm
two_step = summarize_prompt | llm | to_bullets_input | bullets_prompt | llm

paragraphe = (
    "LangChain is a framework for building applications powered by language models. "
    "It provides prompt templates, chains, memory, and integrations with many model "
    "providers. Developers use it to compose multi-step pipelines, connect models to "
    "external data, and prototype quickly without rewriting boilerplate each time."
)

t0 = time.time()
bullets = two_step.invoke({"text": paragraphe})
print("=== 3 points clés ===")
print(bullets)
print(f"\n⏱️ Latence totale (2 étapes) : {time.time()-t0:.2f} s")

In [ ]:
# Voir aussi le résumé intermédiaire seul (utile pour déboguer la chaîne)
intermediate = (summarize_prompt | llm).invoke({"text": paragraphe})
print("Résumé intermédiaire :", intermediate)

---
## Partie 4 (Bonus) : Chaîne de conversation avec mémoire

`ConversationChain` n'existe plus en LangChain 1.x. On reproduit son comportement avec
**`RunnableWithMessageHistory`** (l'approche mémoire moderne) : l'historique est conservé
entre les tours, donc le second message garde le contexte du premier.

> Note : les modèles seq2seq comme flan-t5 ne sont pas des vrais modèles de chat. On garde
> l'exemple volontairement simple pour **illustrer la persistance de la mémoire**, pas la
> qualité conversationnelle.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory

# Style système modifiable (ex : concis et encourageant)
SYSTEM_STYLE = "You are concise and encouraging."

chat_prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_STYLE),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}"),
])

# Store mémoire (une session ici)
_store = {}
def get_history(session_id: str):
    if session_id not in _store:
        _store[session_id] = InMemoryChatMessageHistory()
    return _store[session_id]

conversation = RunnableWithMessageHistory(
    chat_prompt | llm,
    get_history,
    input_messages_key="input",
    history_messages_key="history",
)

cfg = {"configurable": {"session_id": "demo"}}

# Tour 1 : salutation
r1 = conversation.invoke({"input": "Hi! My name is Sam."}, config=cfg)
print("Tour 1 :", r1)

# Tour 2 : question de suivi -> la mémoire doit se souvenir du nom
r2 = conversation.invoke({"input": "What is my name?"}, config=cfg)
print("Tour 2 :", r2)

In [ ]:
# Inspecter la mémoire accumulée -> prouve que le contexte est conservé
print("=== Historique conservé ===")
for msg in _store["demo"].messages:
    print(f"[{msg.type}] {msg.content}")

---
## 📝 Observations (livrable Markdown)

**Latence**
- Chargement du modèle : ~5-15 s au premier appel (téléchargement + init).
- Inférence flan-t5-small sur CPU : ~0,3-2 s par appel court.
- Le pipeline 2 étapes double naturellement la latence (deux passages LLM).

**Qualité**
- flan-t5-small suit correctement les instructions simples (réécrire, résumer) mais reste
  limité : sorties courtes, puces parfois incomplètes ou fusionnées.
- Passer à `flan-t5-base` améliore nettement la cohérence, au prix de plus de RAM/latence.

**Anomalies éventuelles**
- Les modèles seq2seq ne « chattent » pas vraiment : la Partie 4 illustre la **mémoire**,
  pas une conversation naturelle. Pour un vrai chat CPU, essayer `Qwen/Qwen2.5-0.5B-Instruct`
  avec `pipeline('text-generation')`.
- Le nombre de puces demandé (3) n'est pas garanti par un si petit modèle — c'est une limite
  du modèle, pas du pipeline.

---

### 🔁 Équivalence ancienne vs nouvelle API (référence)

| Tutoriel (legacy, retiré en 1.x) | Équivalent moderne utilisé ici |
|---|---|
| `LLMChain(llm, prompt)` | `prompt \| llm` |
| `SimpleSequentialChain([c1, c2])` | `p1 \| llm \| adaptateur \| p2 \| llm` |
| `ConversationChain(memory=...)` | `RunnableWithMessageHistory(...)` |

Si votre cours **impose** l'ancienne API, épinglez d'anciennes versions au tout début :
```python
!pip install -q "langchain==0.1.20" "langchain-community==0.0.38"
```
puis `from langchain.chains import LLMChain, ConversationChain, SimpleSequentialChain`.
Sinon, l'API moderne ci-dessus est recommandée.

---

## ✅ Récapitulatif des livrables
| Partie | Livrable |
|---|---|
| 1 | Packages installés + vérification matériel |
| 2 | Chaîne LLM testée qui réécrit le texte |
| 3 | Pipeline 2 étapes : résumé → 3 points clés |
| 4 | (Bonus) Conversation avec mémoire persistante |
| — | Cellule d'observations (latence / qualité / anomalies) |